In [1]:
!pip install transformers torch nltk rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=03eaa9a7f61fc89772de067b46936e709461ed977b21ef7b70f700c77bd4ab8e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
!pip install rouge-score

In [ ]:
# Pytorch is used to run the model and handle tensor operations.
# Transformers to load and generate output from a pretrained model
# nltk to compute BLEU score
# rouge score to compute ROUGE metrics

In [ ]:
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM,AutoModelForSeq2SeqLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

In [ ]:
# Load a Pretrained Language Model
# This code loads the FLAN-T5 base model for
# sequence to sequence text generation.
# The tokenizer converts text into model ready tokens and
# the model loads its pretrained weights to generate outputs
# for evaluation.

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# Generate Text from the Model
# This code generates text from the pretrained model using a
#given prompt.

# tokenizer converts the prompt into tensors suitable for the model.
# torch.no_grad() disables gradient computation since
#we are only performing inference.
# model.generate() produces output tokens, limited to 80 new tokens.
# tokenizer.decode() converts the generated tokens back into
#readable text.

In [ ]:
prompt = "Explain what is machine learning."

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
       **inputs,
      max_new_tokens=80
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Text:\n", generated_text)

Generated Text:
 Machine learning is a technique for detecting patterns in data.


In [ ]:
# Prepare Reference and Candidate Text
# This code prepares the human written reference and the model generated output for evaluation.

# reference_text represents the ground truth sentence.
# candidate_text contains the text generated by the model.
# Both texts are split into tokens (words) because BLEU requires tokenized input.
# The reference is wrapped inside a list since BLEU expects one or more reference sentences.

In [ ]:
reference_text = "Machine learning is a method where computers learn patterns from data and make predictions without being explicitly programmed."

candidate_text = generated_text

reference_tokens = [reference_text.split()]
candidate_tokens = candidate_text.split()

In [ ]:
# Compute BLEU Score
# This code calculates the BLEU score between the reference text and the model generated output.

# SmoothingFunction().method1 is applied to avoid zero scores when higher order n grams do not match.
# sentence_bleu() compares the tokenized candidate text against the reference tokens.
# The final score reflects how closely the generated output matches the reference in terms of n gram precision.

In [ ]:
smooth = SmoothingFunction().method1

bleu_score = sentence_bleu(
    reference_tokens,
    candidate_tokens,
    smoothing_function=smooth
)

print("BLEU Score:", bleu_score)

BLEU Score: 0.12480646832647957


In [ ]:
# Compute ROUGE Score
# This code evaluates the generated text using ROUGE metrics.

# rouge1 measures unigram overlap.
# rouge2 measures bigram overlap.
# rougeL measures the longest common subsequence similarity.
# use_stemmer=True improves matching by reducing words to their root forms.

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge_scores = scorer.score(reference_text, candidate_text)

for key, value in rouge_scores.items():
    print(f"{key}")
    print(f"Precision: {value.precision:.4f}")
    print(f"Recall: {value.recall:.4f}")
    print(f"F1 Score: {value.fmeasure:.4f}")
    print()

rouge1
Precision: 0.6000
Recall: 0.3333
F1 Score: 0.4286

rouge2
Precision: 0.3333
Recall: 0.1765
F1 Score: 0.2308

rougeL
Precision: 0.6000
Recall: 0.3333
F1 Score: 0.4286

